In [2]:
import numpy as np
from petsc4py import PETSc
from slepc4py import SLEPc


# Function to create PETSc sparse matrix from data
def create_petsc_matrix(data):
    # Extract rows, columns, and values
    rows = data[:, 0].astype(int) - 1  # Convert to zero-based index
    cols = data[:, 1].astype(int) - 1  # Convert to zero-based index
    values = data[:, 2]

    # Create PETSc matrix
    nrows = rows.max() + 1
    ncols = cols.max() + 1
    A = PETSc.Mat().createAIJ([nrows, ncols], nnz=3)
    A.setOption(PETSc.Mat.Option.NEW_NONZERO_ALLOCATION_ERR, False)
    A.setUp()
    for r, c, v in zip(rows, cols, values):
        A[r, c] = v
    A.assemble()

    return A

def compute_invdiag(A):
    diagA = A.getDiagonal()
    diagA_inv = diagA.copy()

    # Compute the lumped row reciprocal of each diagonal entry manually
    for i in range(diagA_inv.size):
        rowsum = 0.0
        for j in range(diagA_inv.size):
            rowsum += A[i, j]
        if rowsum != 0:
            diagA_inv[i] = 1.0 / rowsum
        else:
            print("Row sum is zero for row", i)
            break

    # Create a diagonal matrix D_inv from the inverse diagonal values
    D_inv = PETSc.Mat().createAIJ(A.getSize(), comm=comm)
    D_inv.setUp()
    for i in range(diagA_inv.size):
        D_inv.setValue(i, i, diagA_inv[i])
    D_inv.assemble()
    
    return D_inv

In [3]:
from mpi4py import MPI
import sys

comm = MPI.COMM_WORLD

def par_print(string):
    if comm.rank == 0:
        print(string)
        sys.stdout.flush()

def eigenvalues(A, B, shift=0.0, n_eigs=11, precond = False):
    # Check if need precond
    if precond == True:
        D_inv = compute_invdiag(A+B)
        # D_inv = compute_invdiag(A)
        A = D_inv.matMult(A)
        B = D_inv.matMult(B)


    # Create SLEPc Eigenvalue solver
    eps = SLEPc.EPS().create(comm)
    eps.setOperators(A, B)
    eps.setType(SLEPc.EPS.Type.KRYLOVSCHUR)
    eps.setProblemType(SLEPc.EPS.ProblemType.GHEP)
    eps.setWhichEigenpairs(eps.Which.TARGET_MAGNITUDE)
    eps.setTarget(shift)

    st = eps.getST()
    st.setType(SLEPc.ST.Type.SINVERT)
    st.setShift(shift)

    eps.setDimensions(n_eigs, PETSc.DECIDE, PETSc.DECIDE)
    eps.setFromOptions()
    eps.solve()

    its = eps.getIterationNumber()
    eps_type = eps.getType()
    n_ev, n_cv, mpd = eps.getDimensions()
    tol, max_it = eps.getTolerances()
    n_conv = eps.getConverged()

    par_print(f"Number of iterations: {its}")
    par_print(f"Solution method: {eps_type}")
    par_print(f"Number of requested eigenvalues: {n_ev}")
    par_print(f"Stopping condition: tol={tol}, maxit={max_it}")
    par_print(f"Number of converged eigenpairs: {n_conv}")

    computed_eigenvalues = []
    for i in range(min(n_conv, n_eigs)):
        lmbda = eps.getEigenvalue(i)
        # computed_eigenvalues.append(np.round(np.real(lmbda), 1))
        computed_eigenvalues.append(np.round(np.real(lmbda), 5))

    eps.destroy()

    # Print results
    np.set_printoptions(formatter={'float': '{:5.1f}'.format})
    # eigenvalues_exact = np.sort(np.array([float(m**2 + n**2 + k**2)
    #                                       for m in range(6)
    #                                       for n in range(6)
    #                                       for k in range(6)]))[1:13]
    eigenvalues_exact = np.array([2.0, 2.0, 2.0, 3.0, 3.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0])
    computed_eigenvalues = np.sort(computed_eigenvalues)
    
    par_print(f"Exact    = {eigenvalues_exact}")
    par_print(f"Nédélec  = {computed_eigenvalues}")
    return computed_eigenvalues

In [6]:
# Read the matrix data
# i = 0
# A_data = np.loadtxt("../../build/A0.dat")
# B_data = np.loadtxt("../../build/B0.dat")
# i = 1
A_data = np.loadtxt("../../build/A1.dat")
B_data = np.loadtxt("../../build/B1.dat")

# Create the PETSc matrices A and B
A = create_petsc_matrix(A_data)
B = create_petsc_matrix(B_data)

evals = eigenvalues(A, B, 3.2, 11)
# evals = eigenvalues(A, B, 3.2, 41)
# evals = eigenvalues(A, B, 1.1, 11, True)


# evals = evals[..., None]
np.savetxt(sys.stdout, evals, fmt="%.1f")

Number of iterations: 3
Solution method: krylovschur
Number of requested eigenvalues: 11
Stopping condition: tol=1e-08, maxit=3492
Number of converged eigenpairs: 11
Exact    = [  2.0   2.0   2.0   3.0   3.0   5.0   5.0   5.0   5.0   5.0   5.0]
Nédélec  = [  2.0   2.0   2.0   3.0   3.0   4.8   4.9   4.9   5.0   5.0   5.0]
2.0
2.0
2.0
3.0
3.0
4.8
4.9
4.9
5.0
5.0
5.0


In [146]:
Ad = A.convert('dense').getDenseArray()
Bd = B.convert('dense').getDenseArray()

size = Bd.shape
Bd[size[0]-2,size[1]-2]
Bd[0, 0]

# Ad[1968-1, 1968-1]
# Ad.diagonal()[1968-21 : 1968+21]

8.726646259971649e-14

In [88]:
print(diagA)